# Phase 3: training the relevance model

The robot collected 450 job postings from karriere.at, and I labeled each one by hand:
**1 = relevant** (I would apply), **0 = not relevant**. The criteria are in `labeling/criteria.md`.

This notebook trains a classifier that predicts the label for new postings. It follows the method from
the course unit *ML for RPA on Text Data*:

* bag-of-words with `CountVectorizer` (binary, max. 500 words),
* the classes are balanced as in the course: the same number of relevant and not relevant jobs
  (`groupby().sample()`),
* `MLPClassifier` with two hidden layers (100, 10),
* 80/20 train/test split,
* the trained model is saved as plain CSV/JSON files (the course saves it with `pickle`; plain files can be
  opened in Excel and explained, and the robot computes the prediction from them with numpy).

Output: the folder `model/` (used by the robot).

In [1]:
import json
import os
import re
import warnings

import numpy as np
import pandas as pd
from sklearn.exceptions import ConvergenceWarning
from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.neural_network import MLPClassifier

warnings.filterwarnings('ignore', category=ConvergenceWarning)  # small max_iter on purpose, as in the course
rs = 1                                                           # random_state everywhere -> reproducible
os.makedirs('../model/weights', exist_ok=True)

## 1. Load the data

`jobs_raw.csv` is written by the robot, `labels.csv` by the labeling app. They are joined on `job_id`.
Jobs marked as unsure (`?`) are left out.

In [2]:
jobs = pd.read_csv('../data/jobs_raw.csv', dtype=str).drop(columns='label')
labels = pd.read_csv('../data/labels.csv', dtype=str)
df = jobs.merge(labels[['job_id', 'label']], on='job_id')
df = df[df.label.isin(['0', '1'])].reset_index(drop=True)
df['y'] = df.label.astype(int)
print(df.shape)
df.y.value_counts()

(450, 13)


y
0    300
1    150
Name: count, dtype: int64

In [3]:
# share of relevant jobs per search query
df.groupby('search_query').y.agg(['count', 'sum', 'mean']).sort_values('mean', ascending=False).round(2)

,count,sum,mean
search_query,,,
it-support,48,26,0.54
ai,32,17,0.53
data-analyst,48,21,0.44
digital,42,18,0.43
data-engineer,33,11,0.33
ki,48,16,0.33
business-intelligence,41,12,0.29
data-scientist,15,4,0.27
controlling,43,11,0.26


## 2. Balancing the classes

Only 150 of the 450 jobs (33%) are relevant. As explained in the course, a net trained on unbalanced data can
simply learn to predict the more frequent class and still reach a high accuracy. The course method is used:
the size of the smallest class is determined, and the same number of jobs is drawn from each class
(`groupby().sample()`), then the rows are shuffled.

In [4]:
df_all = df.copy()                       # all 450 labeled jobs
minobs = df.y.value_counts().min()
print('jobs per class:', minobs)
df = df.groupby('y').sample(n=minobs, random_state=rs)
df = df.sample(frac=1, random_state=rs).reset_index(drop=True)   # shuffle
unused = df_all[~df_all.job_id.isin(df.job_id)]                   # 150 not relevant jobs that were not drawn
print(df.shape)
print(df.y.value_counts())

jobs per class: 150
(300, 13)
y
1    150
0    150
Name: count, dtype: int64


From now on the model works with these 300 jobs (150 relevant, 150 not relevant). The 150 not relevant jobs
that were not drawn are not used for training; they are used for an extra check in section 7.

## 3. Which text to use: title or full description?

The description is long (median about 4,000 characters) and mostly describes the company, benefits and the
application process, which is similar in all postings. The title says what the job is.
Quick comparison with the course setup (bag-of-words + MLP), averaged over 10 random 80/20 splits:

In [5]:
GERMAN_STOP_WORDS = '''und oder der die das den dem des ein eine einen einem einer eines mit von für auf aus bei
zum zur ist sind wir sie ihr ihre ihren ihrem uns unser unsere unseren dich dir du sowie als auch wie im in an nach
über um zu nicht werden wird hast hat haben bist sein kann können durch dass es noch bzw mehr sehr alle'''.split()
# English list from sklearn + a small German list (most postings are German); 'not' is kept, as in the course
STOP_WORDS = sorted(w for w in ENGLISH_STOP_WORDS.union(GERMAN_STOP_WORDS) if w != 'not')
TOKEN_PATTERN = r'\b[a-zA-ZäöüÄÖÜß]{3,}\b'   # words with 3+ letters, including German umlauts


def make_vectorizer():
    return CountVectorizer(lowercase=True, binary=True, dtype=np.int8, max_features=500,
                           stop_words=STOP_WORDS, token_pattern=TOKEN_PATTERN)


def evaluate_text(texts, seeds=range(1, 11)):
    scores = []
    for seed in seeds:
        t_tr, t_te, y_tr, y_te = train_test_split(texts, df.y.values, test_size=0.2, random_state=seed, stratify=df.y)
        vctr = make_vectorizer()
        X_tr, X_te = vctr.fit_transform(t_tr), vctr.transform(t_te)
        clf = MLPClassifier(random_state=rs, max_iter=30, hidden_layer_sizes=(100, 10,)).fit(X_tr, y_tr)
        p = clf.predict(X_te)
        scores.append([accuracy_score(y_te, p), precision_score(y_te, p), recall_score(y_te, p), f1_score(y_te, p)])
    return np.round(np.mean(scores, axis=0), 2)


variants = {
    'title': df.title,
    'title + first 800 characters': df.title + ' ' + df.description.str[:800],
    'title + full description': df.title + ' ' + df.description,
}
pd.DataFrame({name: evaluate_text(t) for name, t in variants.items()},
             index=['accuracy', 'precision', 'recall', 'F1']).T

,accuracy,precision,recall,F1
title,0.63,0.62,0.78,0.67
title + first 800 characters,0.55,0.55,0.62,0.58
title + full description,0.57,0.57,0.58,0.57


**Result:** the title alone works clearly best. More text adds words that appear in relevant and
non-relevant postings alike, and with only 300 examples the model cannot learn to ignore them.
The model therefore uses the **job title** only.

## 4. Train/test split

20% of the jobs are kept aside as test set and only used once, for the final evaluation.
`stratify` keeps both classes equal (50/50) in both parts.

In [6]:
X_text = df.title.str.lower()
y = df.y.values
X_tr_text, X_te_text, y_tr, y_te, idx_tr, idx_te = train_test_split(
    X_text, y, df.index.values, test_size=0.2, random_state=rs, stratify=y)
print('train:', len(y_tr), 'relevant:', y_tr.sum(), '| test:', len(y_te), 'relevant:', y_te.sum())

train: 240 relevant: 120 | test: 60 relevant: 30


## 5. Bag-of-words

Same settings as in the course: lowercase, binary (word present or not), max. 500 words, stop words removed,
only words with 3 or more letters. The vectorizer is fitted on the training data only. The 300 titles contain
fewer than 500 different words, so the vocabulary is smaller than `max_features`.

In [7]:
vctr = make_vectorizer()
X_tr = vctr.fit_transform(X_tr_text)
X_te = vctr.transform(X_te_text)
print(X_tr.shape, '- vocabulary size:', len(vctr.vocabulary_))
print(sorted(vctr.vocabulary_)[:40])

(240, 345) - vocabulary size: 345
['abschluss', 'accountant', 'accounting', 'administrator', 'adoption', 'advisor', 'advisory', 'agent', 'agentic', 'aio', 'allround', 'allrounder', 'analyst', 'analytic', 'analytics', 'anwält', 'application', 'applications', 'applied', 'applikationsbetreuer', 'arbeitgeber', 'architect', 'architecture', 'architektur', 'asset', 'associate', 'assurance', 'audit', 'auditor', 'automation', 'automatisierungs', 'awareness', 'azure', 'backend', 'banking', 'bcbs', 'bereich', 'bereichsleitung', 'betreuer', 'betrugsbekämpfung']


## 6. Number of training iterations

In the course `max_iter=5` was enough for 6,400 complaints; with 240 short titles the network needs more passes.
`max_iter` is chosen with 5-fold cross-validation **on the training set only**, so the test set stays unseen
(the course notes that tuning parameters on the test set makes the evaluation optimistic).

In [8]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=rs)
rows = []
for max_iter in [5, 10, 20, 30, 50, 100]:
    f1s = []
    for tr, va in cv.split(X_tr, y_tr):
        clf = MLPClassifier(random_state=rs, max_iter=max_iter, hidden_layer_sizes=(100, 10,)).fit(X_tr[tr], y_tr[tr])
        f1s.append(f1_score(y_tr[va], clf.predict(X_tr[va])))
    rows.append((max_iter, np.mean(f1s)))
cv_results = pd.DataFrame(rows, columns=['max_iter', 'mean F1 (validation)']).round(3)
best_iter = int(cv_results.loc[cv_results['mean F1 (validation)'].idxmax(), 'max_iter'])
print('chosen max_iter:', best_iter)
cv_results

chosen max_iter: 100


,max_iter,mean F1 (validation)
0,5,0.651
1,10,0.640
2,20,0.644
3,30,0.614
4,50,0.640
5,100,0.656


## 7. Training and evaluation on the test set

In [9]:
clf = MLPClassifier(random_state=rs, max_iter=best_iter, hidden_layer_sizes=(100, 10,))
clf.fit(X_tr, y_tr)
print('training score:', round(clf.score(X_tr, y_tr), 3))
print('test score:    ', round(clf.score(X_te, y_te), 3))

p_te = clf.predict(X_te)
prob_te = clf.predict_proba(X_te)[:, 1]
metrics = {
    'accuracy': accuracy_score(y_te, p_te),
    'precision_relevant': precision_score(y_te, p_te),
    'recall_relevant': recall_score(y_te, p_te),
    'f1_relevant': f1_score(y_te, p_te),
}
cm = pd.DataFrame(confusion_matrix(y_te, p_te), index=['true 0', 'true 1'], columns=['predicted 0', 'predicted 1'])
print({k: round(v, 3) for k, v in metrics.items()})
cm

training score: 0.996
test score:     0.667
{'accuracy': 0.667, 'precision_relevant': 0.625, 'recall_relevant': 0.833, 'f1_relevant': 0.714}


,predicted 0,predicted 1
true 0,15,15
true 1,5,25


**How to read it:** *precision* = of the jobs the model calls relevant, how many really are;
*recall* = of the really relevant jobs, how many the model finds. For a job search recall matters more:
a missed good job is worse than one extra job to look at. Because the test set is balanced, random guessing
gives an accuracy of 0.5 (as in the course check: 1/n for n classes).

### Extra check: the 150 not relevant jobs that were not used

These jobs were removed by the balancing in section 2, so the model has never seen them. All of them are not
relevant, so every job the model recommends here is a false positive.

In [10]:
p_unused = clf.predict(vctr.transform(unused.title.str.lower()))
print(f'wrongly recommended: {p_unused.sum()} of {len(unused)} ({p_unused.mean():.0%})')
print(f'correctly rejected:  {(p_unused == 0).sum()} of {len(unused)} ({(p_unused == 0).mean():.0%})')

wrongly recommended: 71 of 150 (47%)
correctly rejected:  79 of 150 (53%)


### Comparison with two simple baselines

In [11]:
import re
rule_pos = re.compile(r'data|daten|analy|\bbi\b|business intelligence|reporting|power bi|consult|\bki\b|\bai\b|machine learning', re.I)
rule_neg = re.compile(r'senior|\blead|head|leit|manager|director|principal|praktik|intern', re.I)
rule = np.array([int(bool(rule_pos.search(t)) and not rule_neg.search(t)) for t in df.title.values[idx_te]])
majority = np.zeros_like(y_te)


def row(pred):
    return [accuracy_score(y_te, pred), precision_score(y_te, pred, zero_division=0),
            recall_score(y_te, pred), f1_score(y_te, pred)]


baselines = pd.DataFrame({
    'always "not relevant"': row(majority),
    'keyword rule on title': row(rule),
    'MLP model': row(p_te),
}, index=['accuracy', 'precision', 'recall', 'F1']).T.round(2)
baselines

,accuracy,precision,recall,F1
"always ""not relevant""",0.50,0.00,0.00,0.00
keyword rule on title,0.63,0.75,0.40,0.52
MLP model,0.67,0.62,0.83,0.71


The keyword rule (data/analyst/BI/reporting in the title, no senior/lead/manager) is what I would write by hand.
The model learns similar rules from the labels, but also words I did not think of.

### Where the model is wrong

In [12]:
test = df.loc[idx_te, ['job_id', 'title', 'company', 'search_query', 'url']].copy()
test['label'] = y_te
test['predicted'] = p_te
test['probability_relevant'] = prob_te.round(3)
print('Relevant jobs the model missed (false negatives):')
display(test[(test.label == 1) & (test.predicted == 0)][['title', 'probability_relevant']])
print('Not relevant jobs the model recommends (false positives):')
display(test[(test.label == 0) & (test.predicted == 1)][['title', 'probability_relevant']])

Relevant jobs the model missed (false negatives):


,title,probability_relevant
187,Marketing Automation Specialist (m/w/d),0.305
22,IT-Entwickler (w/m/d) für Automatisierung und KI,0.102
103,Group IRRD Reporting Riskmanager (m/w/d),0.064
165,AI & Data Consultant - Microsoft Azure (m/w/x),0.084
110,Sales Controller m/w/d – innovatives Handelsun...,0.119


Not relevant jobs the model recommends (false positives):


,title,probability_relevant
262,Bauhelfer (w/m/d),0.677
116,Full Stack Developer (m/w/x),0.506
182,Solution Architect – Steuerungs- & Energiesyst...,0.955
104,Robotics / Physical AI Experte (m/w/d),0.893
275,Release Train Engineer - Agentic AI Platform (...,0.943
141,Spezialist:in Aufsichtsrechliches Meldewesen (...,0.958
4,Junior Projektmanagement Banknote/ technische ...,0.959
50,Marketing- & Fundraising Manager:in (30-40h m/...,0.635
15,Kundenberater:in für die virtuelle Stadtbank (...,0.677
278,Financial Analyst (m/f/d) (Business Controller),0.730


## 8. Final model

For the robot, the model is trained again on **all 300 balanced jobs** (train + test) with the same settings
(more data; the test result above is the estimate of its quality).

In [13]:
vctr_final = make_vectorizer()
X_all = vctr_final.fit_transform(X_text)
clf_final = MLPClassifier(random_state=rs, max_iter=best_iter, hidden_layer_sizes=(100, 10,)).fit(X_all, y)
words = vctr_final.get_feature_names_out()   # column order of the bag-of-words
print('layers (weights):', [w.shape for w in clf_final.coefs_])
print('activation hidden layers:', clf_final.activation, '| output:', clf_final.out_activation_)

layers (weights): [(399, 100), (100, 10), (10, 1)]
activation hidden layers: relu | output: logistic


## 9. Saving the model as plain files

The network is nothing more than 3 weight matrices and 3 bias vectors. They are saved as CSV:

| file | shape | meaning |
|---|---|---|
| `weights/W1.csv`, `b1.csv` | words x 100 | input words -> hidden layer 1 (one row per vocabulary word) |
| `weights/W2.csv`, `b2.csv` | 100 x 10 | hidden layer 1 -> hidden layer 2 |
| `weights/W3.csv`, `b3.csv` | 10 x 1 | hidden layer 2 -> output |

The prediction for a title is:

1. `x` = 1 for every vocabulary word that appears in the lowercased title, else 0 (one value per word)
2. `h1 = relu(x · W1 + b1)`, `h2 = relu(h1 · W2 + b2)`  (relu = max(0, value))
3. `probability = sigmoid(h2 · W3 + b3)`  (sigmoid = 1 / (1 + e^-value))

In [14]:
W1, W2, W3 = clf_final.coefs_
b1, b2, b3 = clf_final.intercepts_
pd.DataFrame(W1, index=words).to_csv('../model/weights/W1.csv', index_label='word')
pd.DataFrame(W2).to_csv('../model/weights/W2.csv', index_label='unit')
pd.DataFrame(W3).to_csv('../model/weights/W3.csv', index_label='unit')
for name, b in [('b1', b1), ('b2', b2), ('b3', b3)]:
    pd.DataFrame(b, columns=['bias']).to_csv(f'../model/weights/{name}.csv', index_label='unit')

model_info = {
    'input': 'job title, lowercased',
    'token_pattern': TOKEN_PATTERN,
    'vocabulary_file': 'vocabulary.csv (column order = row order of W1.csv)',
    'layers': ['W1.csv + b1.csv, relu', 'W2.csv + b2.csv, relu', 'W3.csv + b3.csv, sigmoid'],
    'threshold': 0.5,
    'trained_on_jobs': int(len(y)),
    'sklearn_model': f'MLPClassifier(hidden_layer_sizes=(100, 10), max_iter={best_iter}, random_state={rs})',
}
json.dump(model_info, open('../model/model_info.json', 'w', encoding='utf-8'), indent=2, ensure_ascii=False)
print('saved: model/weights/*.csv, model/model_info.json')

saved: model/weights/*.csv, model/model_info.json


### Check: the model loaded from the CSV files gives the same result as scikit-learn

This is the same code the robot uses (`libraries/JobModel.py`): only the CSV files, numpy and the formula above.

In [15]:
vocab_order = pd.read_csv('../model/weights/W1.csv', usecols=['word']).word.tolist()
L1 = pd.read_csv('../model/weights/W1.csv', index_col='word').values
L2 = pd.read_csv('../model/weights/W2.csv', index_col='unit').values
L3 = pd.read_csv('../model/weights/W3.csv', index_col='unit').values
B1, B2, B3 = (pd.read_csv(f'../model/weights/{n}.csv', index_col='unit').bias.values for n in ['b1', 'b2', 'b3'])
token_re = re.compile(TOKEN_PATTERN)


def predict_from_files(title):
    tokens = set(token_re.findall(title.lower()))
    x = np.array([1.0 if w in tokens else 0.0 for w in vocab_order])
    h1 = np.maximum(0, x @ L1 + B1)
    h2 = np.maximum(0, h1 @ L2 + B2)
    return 1 / (1 + np.exp(-(h2 @ L3 + B3)[0]))


p_files = np.array([predict_from_files(t) for t in df.title])
p_sklearn = clf_final.predict_proba(X_all)[:, 1]
print(f'largest difference over all {len(df)} titles:', np.abs(p_files - p_sklearn).max())

examples = ['Junior Data Analyst (m/w/d)', 'Senior Data Engineer (m/w/d)', 'Verkäufer:in Teilzeit',
            'Business Intelligence Consultant', 'IT Support Techniker (m/w/d)']
pd.DataFrame({'title': examples, 'probability_relevant': [round(predict_from_files(t), 3) for t in examples]})

largest difference over all 300 titles: 2.220446049250313e-16


,title,probability_relevant
0,Junior Data Analyst (m/w/d),0.996
1,Senior Data Engineer (m/w/d),0.023
2,Verkäufer:in Teilzeit,0.435
3,Business Intelligence Consultant,0.860
4,IT Support Techniker (m/w/d),0.992


## 10. Files for the documentation

* `model/vocabulary.csv` - the words of the model and how often each appears in relevant / not relevant titles
* `model/metrics.json` - test metrics, baselines, settings
* `model/confusion_matrix.csv`
* `model/test_predictions.csv` - every test job with my label and the model's probability

In [16]:
in_rel = np.asarray(X_all[y == 1].sum(axis=0)).ravel()
in_not = np.asarray(X_all[y == 0].sum(axis=0)).ravel()
vocab = pd.DataFrame({'column': np.arange(len(words)), 'word': words,
                      'titles_relevant': in_rel, 'titles_not_relevant': in_not})
vocab['share_relevant'] = (vocab.titles_relevant / (vocab.titles_relevant + vocab.titles_not_relevant)).round(2)
vocab = vocab.sort_values(['titles_relevant', 'titles_not_relevant'], ascending=False)
vocab.to_csv('../model/vocabulary.csv', index=False)

info = {
    'trained_at': pd.Timestamp.now().isoformat(timespec='seconds'),
    'data': {'labeled_jobs': int(len(df_all)), 'labeled_relevant': int(df_all.y.sum()),
             'used_after_balancing': int(len(y)), 'relevant': int(y.sum()), 'not_relevant': int((y == 0).sum())},
    'features': 'job title, bag-of-words (CountVectorizer binary, max_features=500, stop words EN+DE)',
    'model': f'MLPClassifier(hidden_layer_sizes=(100, 10), max_iter={best_iter}, random_state={rs})',
    'balancing': 'as in the course: groupby().sample() with the size of the smaller class (150 per class)',
    'extra_check_unused_not_relevant': {'jobs': int(len(unused)), 'wrongly_recommended': int(p_unused.sum())},
    'test_set': {'jobs': int(len(y_te)), 'relevant': int(y_te.sum())},
    'test_metrics': {k: round(v, 3) for k, v in metrics.items()},
    'baselines_on_test': baselines.to_dict(orient='index'),
    'cv_max_iter': cv_results.to_dict(orient='records'),
}
json.dump(info, open('../model/metrics.json', 'w'), indent=2)
cm.to_csv('../model/confusion_matrix.csv')
test.sort_values('probability_relevant', ascending=False).to_csv('../model/test_predictions.csv', index=False)

print('Words that appear most in relevant titles:')
display(vocab[vocab.titles_relevant >= 3].sort_values('share_relevant', ascending=False).head(15))
print('Words that appear most in not relevant titles:')
display(vocab[vocab.titles_not_relevant >= 3].sort_values('share_relevant').head(15))

Words that appear most in relevant titles:


,column,word,titles_relevant,titles_not_relevant,share_relevant
210,210,level,4,0,1.00
98,98,devops,3,0,1.00
358,358,techniker,3,0,1.00
237,237,observability,3,0,1.00
315,315,security,5,1,0.83
114,114,engineering,5,1,0.83
242,242,operations,4,1,0.80
99,99,digital,21,6,0.78
334,334,specialist,19,6,0.76
16,16,analytics,6,2,0.75


Words that appear most in not relevant titles:


,column,word,titles_relevant,titles_not_relevant,share_relevant
132,132,financial,0,6,0.00
306,306,sap,0,4,0.00
321,321,services,0,4,0.00
53,53,buchhalter,0,3,0.00
65,65,communication,0,3,0.00
356,356,teamleiter,0,3,0.00
365,365,train,0,3,0.00
357,357,technical,0,3,0.00
207,207,lead,0,3,0.00
71,71,content,0,3,0.00
